In [1]:
from fastai.vision.all import *
import pandas as pd, json
from functools import partial
import torch



/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
# paths and metadata
path = Path("/kaggle/input/cassava-leaf-disease-classification/")
train_df = pd.read_csv(path / "train.csv")
with open(path / "label_num_to_disease_map.json") as f:
    label_dict = json.load(f)
label_dict = {int(k): v for k, v in label_dict.items()}
df = train_df.set_index("image_id")
labels = df.to_dict()["label"]


def get_label(labels, x):
    return labels[Path(x).name]


def get_data(labels, bs=64, presize=500, resize=384):
    tfms = [
        Rotate(90),
        Warp(magnitude=0.4, p=1.0),
        Zoom(min_zoom=0.9, max_zoom=1.3),
        Brightness(max_lighting=0.5),
        Flip(),
        Contrast(),
        Resize(resize),
    ]
    comp = setup_aug_tfms(tfms)
    return DataBlock(
        blocks=(ImageBlock, CategoryBlock),
        get_items=lambda p: get_image_files(p),
        get_y=partial(get_label, labels),
        splitter=RandomSplitter(),
        item_tfms=[Resize(presize)],
        batch_tfms=[*comp, Normalize.from_stats(*imagenet_stats)],
    ).dataloaders(path / "train_images", bs=bs, num_workers=4)




In [3]:
# create dataloaders, model, and train
dls = get_data(labels, bs=64, presize=384)
learn = cnn_learner(dls, models.resnet50, metrics=accuracy, pretrained=True)
learn.fine_tune(3, base_lr=1e-3)

# inference with test‑time augmentation
test_files = get_image_files(path / "test_images")
test_dl = dls.test_dl(test_files)
tta_preds, _ = learn.tta(dl=test_dl, n=5)
# direct argmax over class dimension
preds = tta_preds.argmax(dim=1)



/usr/local/lib/python3.11/dist-packages/fastai/vision/learner.py:303: UserWarning: `cnn_learner` has been renamed to `vision_learner` -- please update your code
  warn("`cnn_learner` has been renamed to `vision_learner` -- please update your code")
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 113MB/s]


epoch,train_loss,valid_loss,accuracy,time
0,1.178663,0.933773,0.691774,01:26


epoch,train_loss,valid_loss,accuracy,time
0,0.791038,0.587966,0.808761,01:41
1,0.616935,0.474080,0.841346,01:41
2,0.545900,0.460247,0.840011,01:41


In [4]:
# build and save submission
test_ids = [f.name for f in test_files]
submission = pd.DataFrame({"image_id": test_ids, "label": preds.cpu().numpy()})
submission.to_csv("submission.csv", index=False)